# Treino U-Net no Kaggle (GPU) — estrutura **roof/** e dois modelos

O notebook usa a pasta **roof/** com:
- **roof/chips/** — imagens + máscaras binárias (Inria + RoofSat, nomes padronizados) → pré-treino
- **roof/chips_multiclass/** — imagens + máscaras multiclasse → fine-tuning
- **roof/chips_segmentos/** — imagens + .npz (e máscaras de linhas geradas) → modelo de linhas das águas

**No final tens dois modelos:**
1. **U-Net multiclasse** (`unet_roof_multiclass.pt`): segmentação em classes (telhado, águas, claraboia, divisória, laje).
2. **U-Net linhas** (`unet_lines.pt`): mapa de linhas (contornos/águas do telhado); em inferência podes extrair segmentos com Hough.

**Só precisas de:** Settings → GPU + Internet On; anexar os datasets (roof-chips-multiclass, roof-inria-patches, roof-roofsat) ou um dataset único com a pasta **roof/** já montada. **Run All**.

In [ ]:
# Verificar GPU — se aparecer "Nenhuma GPU", ativa GPU em Settings → Accelerator (e verifica Phone verification na conta)
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
USE_GPU = (out.returncode == 0)
if USE_GPU:
    print('GPU:', out.stdout.strip())
else:
    print('Nenhuma GPU. O treino vai usar CPU (muito mais lento).')
    print('Para ativar GPU: Settings → Accelerator → GPU. Conta Kaggle precisa de Phone verification.')

In [ ]:
# Clone do repositório roofArea (caminho absoluto para não aninhar ao correr a célula várias vezes)
import os
REPO_URL = "https://github.com/phmotad/roofArea.git"
REPO_DIR = "/kaggle/working/roofArea"

if os.path.isfile(os.path.join(REPO_DIR, "scripts", "train_unet.py")):
    print("Projeto já presente em", REPO_DIR)
else:
    if os.path.isdir(REPO_DIR):
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Diretório atual:", os.getcwd())

In [ ]:
# Instalar dependências
subprocess.run(["pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "pillow", "numpy", "opencv-python-headless", "scikit-image", "huggingface_hub"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

**Caminhos:** Detetamos em `/kaggle/input/` uma pasta **roof/** (chips + chips_multiclass + chips_segmentos) ou os datasets separados. Se não existir roof/, construímos em `/kaggle/working/roof` a partir dos datasets.

In [ ]:
# Caminhos — deteção automática em /kaggle/input/ (até 3 níveis: roofarea/roof-chips-multiclass/chips_multiclass)
import os
KAGGLE_INPUT = "/kaggle/input"

def _has_subdirs(root, *subdirs):
    return all(os.path.isdir(os.path.join(root, d)) for d in subdirs)

def _has_files_in(dirpath, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(dirpath):
        return False
    try:
        names = os.listdir(dirpath)
        return any(f.lower().endswith(ext) for f in names[:50] for ext in exts)
    except OSError:
        return False

# Recolher raizes até 3 níveis (ex.: input/roofarea, input/roofarea/roof-chips-multiclass, input/.../chips_multiclass)
roots_with_name = []
def add_roots(base_path, depth=0):
    if depth > 2:
        return
    try:
        for name in os.listdir(base_path):
            path = os.path.join(base_path, name)
            if os.path.isdir(path):
                roots_with_name.append((path, name.lower()))
                add_roots(path, depth + 1)
    except OSError:
        pass
try:
    add_roots(KAGGLE_INPUT)
except OSError:
    pass

CHIPS_PATH = ""
INRIA_PATH = ""
ROOFSAT_PATH = ""
ROOF_DIR = ""
BUILD_ROOF = True

# Preferir pasta roof/ já montada (dataset único com roof/chips + roof/chips_multiclass)
for root, _ in roots_with_name:
    if _has_subdirs(root, "chips", "chips_multiclass") and _has_files_in(os.path.join(root, "chips", "images")) and _has_files_in(os.path.join(root, "chips_multiclass", "images")):
        ROOF_DIR = root
        CHIPS_PATH = os.path.join(ROOF_DIR, "chips_multiclass")
        INRIA_PATH = os.path.join(ROOF_DIR, "chips")
        SEG = os.path.join(ROOF_DIR, "chips_segmentos")
        ROOFSAT_PATH = SEG if os.path.isdir(SEG) and _has_files_in(os.path.join(SEG, "images")) else ""
        BUILD_ROOF = False
        break

def _check_chips(root):
    cand = os.path.join(root, "chips_multiclass") if os.path.isdir(os.path.join(root, "chips_multiclass")) else root
    return cand if _has_subdirs(cand, "images", "masks") and _has_files_in(os.path.join(cand, "images")) else ""
def _check_inria(root, exclude_path=""):
    inria_cand = os.path.join(root, "dados_inria") if os.path.isdir(os.path.join(root, "dados_inria")) else root
    if _has_subdirs(inria_cand, "images", "masks") and _has_files_in(os.path.join(inria_cand, "images")) and inria_cand != exclude_path:
        return inria_cand
    return ""
def _check_roofsat(root):
    rs = os.path.join(root, "Roofsat") if os.path.isdir(os.path.join(root, "Roofsat")) else root
    if os.path.isdir(os.path.join(rs, "building_masks")) and os.path.isfile(os.path.join(rs, "train.txt")) and (os.path.isdir(os.path.join(rs, "img_color")) or os.path.isdir(os.path.join(rs, "img"))):
        return rs
    return ""

if not ROOF_DIR:
    # Deteção por nome e estrutura (datasets separados ou fallback)
    for root, name in roots_with_name:
        if ("chip" in name or "multiclass" in name) and not CHIPS_PATH:
            c = _check_chips(root)
            if c: CHIPS_PATH = c
        if "inria" in name and not INRIA_PATH:
            i = _check_inria(root, CHIPS_PATH)
            if i: INRIA_PATH = i
        if "roofsat" in name and not ROOFSAT_PATH:
            r = _check_roofsat(root)
            if r: ROOFSAT_PATH = r

    for root, _ in roots_with_name:
        if not CHIPS_PATH:
            c = _check_chips(root)
            if c: CHIPS_PATH = c
        if not INRIA_PATH:
            i = _check_inria(root, CHIPS_PATH)
            if i: INRIA_PATH = i
        if not ROOFSAT_PATH:
            r = _check_roofsat(root)
            if r: ROOFSAT_PATH = r

    try:
        for top in os.listdir(KAGGLE_INPUT):
            base = os.path.join(KAGGLE_INPUT, top)
            if not os.path.isdir(base):
                continue
            if not CHIPS_PATH:
                p = os.path.join(base, "roof-chips-multiclass", "chips_multiclass")
                if os.path.isdir(p) and _has_subdirs(p, "images", "masks"):
                    CHIPS_PATH = p
            if not INRIA_PATH:
                p = os.path.join(base, "roof-inria-patches")
                if _has_subdirs(p, "images", "masks") and _has_files_in(os.path.join(p, "images")):
                    INRIA_PATH = p
            if not ROOFSAT_PATH:
                p = os.path.join(base, "roof-roofsat", "Roofsat")
                if os.path.isdir(p) and os.path.isdir(os.path.join(p, "building_masks")) and os.path.isfile(os.path.join(p, "train.txt")):
                    ROOFSAT_PATH = p
    except OSError:
        pass

    ROOF_DIR = "/kaggle/working/roof"
    BUILD_ROOF = True

if not CHIPS_PATH:
    try:
        print("Pastas em /kaggle/input:", os.listdir(KAGGLE_INPUT))
    except Exception:
        pass
    raise SystemExit("Nenhum dataset de chips em /kaggle/input/. Anexa roof-chips-multiclass ou dataset com pasta roof/. Em Add data adiciona o dataset.")
print("Chips:", CHIPS_PATH)
if INRIA_PATH:
    print("Inria: anexado →", INRIA_PATH)
else:
    INRIA_PATH = "/kaggle/working/dados_inria"
    print("Inria: será descarregado →", INRIA_PATH)
if ROOFSAT_PATH:
    print("RoofSat: anexado →", ROOFSAT_PATH)
else:
    print("RoofSat: não anexado")
print("ROOF_DIR:", ROOF_DIR, "(construir?)" if BUILD_ROOF else "(já existe)")

In [ ]:
# Construir roof/ a partir dos datasets (se ROOF_DIR não existia)
import os
import subprocess
if BUILD_ROOF:
    inria_for_roof = INRIA_PATH
    if not os.path.isdir(os.path.join(INRIA_PATH, "images")) or not os.path.isdir(os.path.join(INRIA_PATH, "masks")):
        inria_for_roof = "/kaggle/working/dados_inria"
        print("A descarregar Inria para", inria_for_roof, "...")
        subprocess.run(["python", "-m", "scripts.download_inria_dataset", "--output_dir", inria_for_roof], check=True)
    cmd = ["python", "-m", "scripts.prepare_roof_structure", "--output", ROOF_DIR, "--inria", inria_for_roof, "--chips_multiclass", CHIPS_PATH]
    if ROOFSAT_PATH:
        cmd.extend(["--roofsat", ROOFSAT_PATH])
    subprocess.run(cmd, check=True)
else:
    print("roof/ já presente; a ignorar construção.")

In [ ]:
# Pré-treino binário (roof/chips = Inria + RoofSat com nomes padronizados)
import subprocess
import os
os.makedirs("/kaggle/working/models", exist_ok=True)
CHIPS_BIN = os.path.join(ROOF_DIR, "chips")
if os.path.isdir(os.path.join(CHIPS_BIN, "images")) and os.path.isdir(os.path.join(CHIPS_BIN, "masks")):
    subprocess.run([
        "python", "-u", "-m", "scripts.train_unet",
        "--data_dir", CHIPS_BIN,
        "--output", "/kaggle/working/models/unet_roof_pretrain.pt",
        "--num_classes", "1", "--size", "512", "512", "--epochs", "30", "--device", "cuda" if USE_GPU else "cpu"
    ], check=True)
else:
    print("roof/chips sem images/masks; a ignorar pré-treino.")

**Fine-tuning multiclasse** em `roof/chips_multiclass` (telhado, águas, claraboia, divisória, laje). Usa o pré-treino binário.

In [ ]:
# Fine-tuning multiclasse (roof/chips_multiclass)
import subprocess
import os
CHIPS_MC = os.path.join(ROOF_DIR, "chips_multiclass")
PRETRAIN_PATH = "/kaggle/working/models/unet_roof_pretrain.pt"
cmd = [
    "python", "-u", "-m", "scripts.train_unet",
    "--data_dir", CHIPS_MC,
    "--output", "/kaggle/working/models/unet_roof_multiclass.pt",
    "--num_classes", "5", "--size", "512", "512", "--epochs", "30", "--device", "cuda" if USE_GPU else "cpu"
]
if os.path.isfile(PRETRAIN_PATH):
    cmd.extend(["--pretrain", PRETRAIN_PATH])
subprocess.run(cmd, check=True)

In [ ]:
# Modelo de linhas (roof/chips_segmentos): gerar máscaras a partir dos .npz e treinar
import subprocess
import os
SEG_DIR = os.path.join(ROOF_DIR, "chips_segmentos")
if os.path.isdir(os.path.join(SEG_DIR, "gt")) and os.path.isdir(os.path.join(SEG_DIR, "images")):
    subprocess.run(["python", "-m", "scripts.rasterize_chips_segmentos_masks", "--segmentos_dir", SEG_DIR], check=True)
masks_seg = os.path.join(SEG_DIR, "masks")
if os.path.isdir(masks_seg) and len(os.listdir(masks_seg)) > 0:
    subprocess.run([
        "python", "-u", "-m", "scripts.train_unet",
        "--data_dir", SEG_DIR,
        "--output", "/kaggle/working/models/unet_lines.pt",
        "--num_classes", "1", "--size", "512", "512", "--epochs", "40", "--device", "cuda" if USE_GPU else "cpu"
    ], check=True)
else:
    print("chips_segmentos sem máscaras (ou sem .npz); modelo de linhas ignorado.")

**Guardar os modelos:** Save Version com **Save output** ativado. Ficam em Output: **unet_roof_multiclass.pt** (segmentação multiclasse) e **unet_lines.pt** (mapa de linhas das águas), prontos para usar na API ou inferência.

In [ ]:
# Listar modelos em /kaggle/working/models (multiclasse + linhas)
import os
for f in sorted(os.listdir("/kaggle/working/models")):
    path = os.path.join("/kaggle/working/models", f)
    print(f, "—", os.path.getsize(path) // 1024, "KB")
print("\nUso: unet_roof_multiclass.pt = segmentação (classes); unet_lines.pt = linhas das águas (mapa → Hough para segmentos).")